# 01. Domain Feature Engineering & Leakage-Free Pipeline
### Track: Data Science (Feature Rationale & Preprocessing)

**Tujuan:**
1. Memformulasi fitur bisnis baru (*Domain Features*) yang terinspirasi dari temuan *Track Analytics*.
2. Memastikan pemisahan data (*Train-Test Split*) dilakukan **SEBELUM** preprocessing untuk menjamin **Zero Data Leakage**.
3. Membangun modul pipeline Scikit-learn (`DomainFeatureEngineer` dan `ColumnTransformer`).
4. Menguji konsistensi dimensi matriks fitur sebelum model training.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "02_prediction" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.telco_churn.core.loader import load_processed_data
from src.telco_churn.prediction.pipelines import (
    DomainFeatureEngineer,
    build_preprocessor,
    get_feature_columns
)


## 1. Membaca Data Bersih & Pemisahan Fitur-Target
Data dibaca dari satu-satunya sumber terverifikasi: `data/processed/telco_churn_clean.csv`.


In [ ]:
df = load_processed_data()
print(f"Data shape: {df.shape}")

# Target biner: 1 jika Churn, 0 jika Tidak
y = (df["churn"] == "Yes").astype(int)
X = df.drop(columns=["churn", "customer_id"])

print("Distribusi Target:")
print(y.value_counts(normalize=True).round(4) * 100)


## 2. Train-Test Split (Strict Anti-Leakage Boundary)
Split dilakukan dengan stratifikasi target (`stratify=y`) untuk menjaga rasio imbalanced class tetap proporsional.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Ukuran Data Train: {X_train.shape[0]} baris")
print(f"Ukuran Data Test:  {X_test.shape[0]} baris")


## 3. Rationale Rekayasa Fitur Domain (*Domain Feature Engineering*)
Berdasarkan insight analytics, kita merekayasa 4 fitur sinyal tinggi:
1. `is_month_to_month`: Menandai pelanggan tanpa komitmen kontrak jangka panjang.
2. `is_electronic_check`: Menandai metode pembayaran paling rawan gagal bayar / churn.
3. `fiber_no_tech_support`: Fitur interaksi segmen bahaya tinggi.
4. `monthly_to_total_ratio`: Rasio beban penagihan bulanan terhadap total akumulasi riwayat.


In [ ]:
engineer = DomainFeatureEngineer()
X_train_engineered = engineer.transform(X_train)
X_train_engineered[["tenure", "is_month_to_month", "is_electronic_check", "fiber_no_tech_support", "monthly_to_total_ratio"]].head()


## 4. Pipeline Scikit-learn Preprocessor
Mengombinasikan:
- `StandardScaler` untuk seluruh variabel numerik.
- `OneHotEncoder(handle_unknown='ignore', drop='first')` untuk variabel kategorikal.


In [ ]:
preprocessor = build_preprocessor()
preprocessor.fit(X_train_engineered)

# Transformasi train set dan test set
X_train_processed = preprocessor.transform(X_train_engineered)

X_test_engineered = engineer.transform(X_test)
X_test_processed = preprocessor.transform(X_test_engineered)

print(f"Bentuk Matriks Akhir X_train: {X_train_processed.shape}")
print(f"Bentuk Matriks Akhir X_test:  {X_test_processed.shape}")


## Ringkasan Verifikasi
- Pipeline bebas dari kebocoran data (*data leakage*): Preprocessor di-fit semata-mata pada data training.
- Matriks fitur siap digunakan oleh modul pelatihan model di `notebooks/02_prediction/02_model_experimentation.ipynb`.
